---

# **[실습] 주택청약 FAQ 시스템 구현**

### **문제 설명**
이전 코드를 기반으로 주택청약 FAQ 시스템을 다음 요구사항에 맞춰 개선합니다. 

1. 응답 품질 향상 (1개 이상)
   - 생성된 답변의 품질을 평가 (답변이 불충분한 경우 예외 처리)
   - 관련성 높은 FAQ 문서 검색 (임베딩 모델, 청크 크기, 벡터 검색 방법 등)

2. 사용자 경험 개선 (1개 이상)
   - 대화 이력 관리 기능 추가 (요약, 트리밍 기능 등 고려)
   - 최근 대화 기반 컨텍스트 구성 
   - 사용자 프로필 기반 맞춤 응답

### **제약 조건**
- Gradio ChatInterface 사용
- RAG 구조 유지

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import os
import json
from pprint import pprint

## step 0) 벡터 저장소 준비

- PRJ01_W3_006에서 전처리된 JSON 파일을 사용
- OpenAI 임베딩으로 Chroma DB 구축 (최초 1회만 실행)

In [3]:
# 문서 로드
from langchain_core.documents import Document

output_file = ".././data/housing_faq_formatted.json"

with open(output_file, 'r', encoding='utf-8-sig') as f:
    formatted_docs = [Document(**doc) for doc in json.load(f)]

print(f"로드된 문서 수: {len(formatted_docs)}")
print(formatted_docs[0].page_content)

로드된 문서 수: 50
[1]
질문: 경기도 과천시에서 공급되는 주택의 해당 주택건설지역의 범위는?
답변: 해당 주택건설지역이란 특별시ㆍ광역시ㆍ특별자치시ㆍ특별자치도(관할 구역 안에 지방자치단체인 시ㆍ군이 없는 특별자치도를 말한다) 또는 시ㆍ군의 행정구역을 말합니다. 따라서, 경기도 과천시에서 공급하는 주택의 경우 과천시가 해당 주택건설지역에 해당됩니다. 참고로, 서울특별시에서 공급되는 주택의 경우 서울특별시 전역, 인천광역시의 경우 인천광역시 전역이 해당 주택건설지역에 해당됩니다.



In [7]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 문서 벡터 저장 (최초 1회)
vector_store = Chroma.from_documents(
    documents=formatted_docs,
    embedding=embeddings,
    collection_name="housing_faq_db_homework",
    persist_directory="./chroma_db",
)
print(f"저장된 문서 수: {vector_store._collection.count()}")

저장된 문서 수: 50


In [9]:
vector_store._collection.count()

50

In [8]:
# vector_store.delete_collection()

## step 1) RAG 시스템 + UI 컨트롤

### 개선 사항
- `gr.Blocks`로 UI 컨트롤 패널 추가
  - 임베딩 모델 선택 (OpenAI / HuggingFace / Ollama)
  - 벡터 저장소 선택 (Chroma / FAISS)
  - 검색 방식 선택 (similarity / mmr)
  - Top-K 슬라이더
  - 유사도 임계값 슬라이더
- `search_documents()`에서 `self.retriever` 사용 (버그 수정)
- `update_retriever()` 메서드 추가 → 설정 변경 시 retriever 교체

In [12]:
import gradio as gr
import json
from typing import List, Optional, Generator
from dataclasses import dataclass

from langchain_chroma import Chroma
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.documents import Document
from pydantic import BaseModel, Field
from langchain_core.language_models import BaseChatModel
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate


# ============================================================
# 데이터 클래스
# ============================================================

class AnswerQualityOutput(BaseModel):
    """LLM 기반 답변 품질 평가 결과 스키마"""
    is_sufficient: bool = Field(description="답변이 질문에 충분히 답하고 있으면 True, 불충분(근거없음/모름 등)이면 False")
    reason: str = Field(description="평가 이유를 한 문장으로")


@dataclass
class SearchResult:
    context: str
    source_documents: Optional[List]


# ============================================================
# RAGSystem 클래스
# ============================================================

class RAGSystem:
    def __init__(
            self,
            llm: BaseChatModel,
            eval_llm: BaseChatModel,
            retriever
        ):
        self.llm = llm or ChatOpenAI(model="gpt-4.1-mini", temperature=0)
        self.eval_llm = eval_llm or ChatOpenAI(model="gpt-4.1-mini", temperature=0)

        if not retriever:
            raise ValueError("검색기(retriever)가 필요합니다.")
        self.retriever = retriever

    def update_retriever(self, retriever):
        """설정 변경 시 retriever를 교체"""
        self.retriever = retriever

    def _format_docs(self, docs: List) -> str:
        return "\n\n".join(doc.page_content for doc in docs)

    def _format_source_documents(self, docs: Optional[List]) -> str:
        if not docs:
            return "\n\nℹ️ 관련 문서를 찾을 수 없습니다."

        formatted_docs = []
        for i, doc in enumerate(docs, 1):
            metadata = doc.metadata if hasattr(doc, 'metadata') else {}
            source_info = []

            if 'question_id' in metadata:
                source_info.append(f"ID: {metadata['question_id']}")
            if 'keyword' in metadata:
                source_info.append(f"키워드: {metadata['keyword']}")
            if 'summary' in metadata:
                source_info.append(f"요약: {metadata['summary']}")

            formatted_docs.append(
                f"📚 참조 문서 {i}\n"
                f"• {' | '.join(source_info) if source_info else '출처 정보 없음'}\n"
                f"• 내용: {doc.page_content}"
            )

        return "\n\n" + "\n\n".join(formatted_docs)

    def _evaluate_answer_quality(self, question: str, answer: str) -> AnswerQualityOutput:
        """생성된 답변의 품질을 LLM으로 평가 (with_structured_output + Pydantic)"""
        prompt = ChatPromptTemplate.from_messages([
            ("system", """생성된 답변이 질문에 충분히 답하고 있는지 평가하세요.

다음 기준으로 판단하세요:
1. 답변이 질문의 핵심을 구체적으로 다루고 있는가?
2. '근거 없음', '잘 모르겠습니다' 등 불충분한 표현만 있지는 않은가?
3. 문서에서 논리적으로 추론 가능한 답변을 제공하고 있는가?"""),
            ("human", "질문: {question}\n\n답변: {answer}")
        ])
        chain = prompt | self.eval_llm.with_structured_output(AnswerQualityOutput)
        return chain.invoke({"question": question, "answer": answer})

    def _check_relevance(self, docs: List, question: str) -> List:
        """검색된 문서가 질문과 관련이 있는지 LLM으로 평가 (Yes/No)"""
        if not docs:
            return []

        prompt = ChatPromptTemplate.from_messages([
            ("system", """주어진 컨텍스트가 질문에 답변하는데 필요한 정보를 포함하고 있는지 평가하세요.

다음 기준 중 하나 이상을 충족할 경우 'Yes'로 답변하고, 모두 충족하지 못하면 'No'로 답변하세요:

1. 컨텍스트가 질문에 답변하는데 필요한 정보를 직접적으로 포함하고 있는가?
2. 컨텍스트의 정보로부터 답변에 필요한 내용을 논리적으로 추론할 수 있는가?

'Yes' 또는 'No'로만 답변하세요."""),
            ("human", """[컨텍스트]
{context}

[질문]
{question}""")
        ])

        chain = prompt | self.eval_llm | StrOutputParser()

        relevant_docs = []
        for doc in docs:
            result = chain.invoke({
                "context": doc.page_content,
                "question": question
            }).lower()

            q_id = doc.metadata.get('question_id', '?')
            print(f"문서 {q_id} 관련성: {result}")

            if "yes" in result:
                relevant_docs.append(doc)

        return relevant_docs

    def search_documents(self, question: str) -> SearchResult:
        try:
            # ✅ self.retriever 사용 (버그 수정: 전역 retriever → self.retriever)
            docs = self.retriever.invoke(question)
            print(f"검색된 문서 개수: {len(docs)}")

            relevant_docs = self._check_relevance(docs, question)
            print(f"관련 문서 개수: {len(relevant_docs)}")

            return SearchResult(
                context=self._format_docs(relevant_docs) if relevant_docs else "관련 문서를 찾을 수 없습니다.",
                source_documents=relevant_docs,
            )
        except Exception as e:
            print(f"문서 검색 중 오류 발생: {e}")
            return SearchResult(
                context="문서 검색 중 오류가 발생했습니다.",
                source_documents=None,
            )

    def generate_answer(self, message: str, history: List) -> Generator[str, None, None]:
        """Gradio 스트리밍 출력을 위한 제너레이터 함수"""

        # 1. 문서 검색 및 관련성 평가
        search_result = self.search_documents(message)

        if not search_result.source_documents:
            yield "죄송합니다. 관련 문서를 찾을 수 없어 답변하기 어렵습니다. 다른 질문을 해주시겠습니까?"
            return

        # 2. 프롬프트 템플릿
        prompt = ChatPromptTemplate.from_messages([
            ("system", """다음 지침을 따라 질문에 답변해주세요:
1. 주어진 문서의 내용만을 기반으로 답변하세요.
2. 문서에 명확한 근거가 없는 내용은 "근거 없음"이라고 답변하세요.
3. 답변하기 어려운 질문은 "잘 모르겠습니다"라고 답변하세요.
4. 추측이나 일반적인 지식을 사용하지 마세요."""),
            ("human", "문서들:\n{context}\n\n질문: {question}")
        ])

        # 3. RAG Chain
        chain = prompt | self.llm | StrOutputParser()

        full_answer = ""
        try:
            # 4. 스트리밍 실행
            for chunk in chain.stream({
                "context": search_result.context,
                "question": message
            }):
                full_answer += chunk
                yield full_answer

            # 5. 답변 품질 평가
            quality = self._evaluate_answer_quality(message, full_answer)

            # 6. 완료 후 참조 문서 추가
            sources = self._format_source_documents(search_result.source_documents)

            if not quality.is_sufficient:
                # 불충분 → 생성된 답변 버리고 폴백 메시지로 교체
                print(f"[품질 평가] 불충분 → 폴백 적용 - {quality.reason}")
                fallback = (
                    "제공된 FAQ 문서에서 질문에 대한 직접적인 답변을 찾지 못했습니다.\n"
                    f"*(평가 이유: {quality.reason})*\n\n"
                    "아래 참조 문서의 관련 내용을 직접 확인해 주세요."
                )
                yield f"{fallback}\n\n---\n{sources}"
            else:
                # 충분 → 원래 답변 그대로 출력
                print(f"[품질 평가] 충분 - {quality.reason}")
                yield f"{full_answer}\n\n---\n{sources}"

        except Exception as e:
            yield f"답변 생성 중 오류가 발생했습니다: {str(e)}"


# ============================================================
# 임베딩 모델 & 벡터 저장소 빌더
# ============================================================

def build_embedding_model(model_name: str):
    """선택된 임베딩 모델 인스턴스 반환"""
    if model_name == "OpenAI text-embedding-3-small":
        return OpenAIEmbeddings(model="text-embedding-3-small")
    elif model_name == "HuggingFace BAAI/bge-m3":
        from langchain_community.embeddings import HuggingFaceEmbeddings
        return HuggingFaceEmbeddings(model_name="BAAI/bge-m3")
    elif model_name == "Ollama bge-m3":
        from langchain_community.embeddings import OllamaEmbeddings
        return OllamaEmbeddings(model="bge-m3")
    else:
        return OpenAIEmbeddings(model="text-embedding-3-small")


def build_retriever(
    embedding_model_name: str,
    vector_store_type: str,
    search_type: str,
    top_k: int,
    similarity_threshold: float
):
    """선택된 설정에 따라 retriever 생성"""
    embeddings = build_embedding_model(embedding_model_name)

    # 벡터 저장소 선택
    if vector_store_type == "Chroma":
        vs = Chroma(
            collection_name="housing_faq_db_homework",
            persist_directory="./chroma_db",
            embedding_function=embeddings,
        )
    else:  # FAISS
        doc_file = ".././data/housing_faq_formatted.json"
        with open(doc_file, "r", encoding="utf-8-sig") as f:
            docs = [Document(**doc) for doc in json.load(f)]
        vs = FAISS.from_documents(docs, embeddings)

    # 검색 방식 선택
    if similarity_threshold > 0:
        # 임계값이 있으면 유사도 임계값 기반 검색
        retriever = vs.as_retriever(
            search_type="similarity_score_threshold",
            search_kwargs={"k": top_k, "score_threshold": similarity_threshold}
        )
    elif search_type == "mmr":
        retriever = vs.as_retriever(
            search_type="mmr",
            search_kwargs={"k": top_k, "fetch_k": top_k * 3, "lambda_mult": 0.5}
        )
    else:  # similarity
        retriever = vs.as_retriever(
            search_type="similarity",
            search_kwargs={"k": top_k}
        )

    return retriever


# ============================================================
# RAGSystem 초기화 (기본값)
# ============================================================

llm = ChatOpenAI(model="gpt-4.1-nano", temperature=0)
eval_llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

default_retriever = build_retriever(
    embedding_model_name="OpenAI text-embedding-3-small",
    vector_store_type="Chroma",
    search_type="similarity",
    top_k=3,
    similarity_threshold=0.0
)

rag_system = RAGSystem(llm=llm, eval_llm=eval_llm, retriever=default_retriever)


# ============================================================
# 설정 적용 함수
# ============================================================

# def apply_settings(embedding_model_name, vector_store_type, search_type, top_k, similarity_threshold):
#     """UI에서 설정 변경 시 retriever 재생성"""
#     try:
#         new_retriever = build_retriever(
#             embedding_model_name=embedding_model_name,
#             vector_store_type=vector_store_type,
#             search_type=search_type if similarity_threshold == 0 else "similarity_score_threshold",
#             top_k=int(top_k),
#             similarity_threshold=float(similarity_threshold)
#         )
#         rag_system.update_retriever(new_retriever)

#         actual_search = search_type if similarity_threshold == 0 else f"similarity_score_threshold (임계값: {similarity_threshold})"
#         return (
#             f"✅ 설정 적용 완료!\n"
#             f"임베딩 모델: {embedding_model_name}\n"
#             f"벡터 저장소: {vector_store_type}\n"
#             f"검색 방식: {actual_search}\n"
#             f"Top-K: {int(top_k)}"
#         )
#     except Exception as e:
#         return f"❌ 설정 오류: {str(e)}"


# ============================================================
# Gradio UI (gr.Blocks)
# ============================================================
# chat 함수가 추가 인자를 직접 받아요
_last_settings = {}  # 이전 설정 기억

def chat_with_settings(message, history, embedding_model, vector_store_type, search_type, top_k, threshold):
    global _last_settings
    
    current_settings = {
        "embedding": embedding_model, "vs": vector_store_type,
        "search": search_type, "k": top_k, "threshold": threshold
    }
    
    # 설정이 바뀐 경우에만 retriever 재생성
    if current_settings != _last_settings:
        retriever = build_retriever(embedding_model, vector_store_type, search_type, int(top_k), threshold)
        rag_system.update_retriever(retriever)
        _last_settings = current_settings
    
    yield from rag_system.generate_answer(message, history)
    
demo = gr.ChatInterface(
    fn=chat_with_settings,
    title="🏠 주택청약 FAQ 챗봇",
    additional_inputs=[
        gr.Dropdown(
            choices=["OpenAI text-embedding-3-small", "HuggingFace BAAI/bge-m3", "Ollama bge-m3"],
            value="OpenAI text-embedding-3-small",
            label="🤖 임베딩 모델"
        ),
        gr.Dropdown(
            choices=["Chroma", "FAISS"],
            value="Chroma",
            label="🗄️ 벡터 저장소"
        ),
        gr.Dropdown(
            choices=["similarity", "mmr"],
            value="similarity",
            label="🔍 검색 방식"
        ),
        gr.Slider(minimum=1, maximum=10, value=3, step=1, label="📦 Top-K"),
        gr.Slider(minimum=0.0, maximum=1.0, value=0.0, step=0.05, label="📏 유사도 임계값"),
    ],
    examples=[
        ["수원시의 주택건설지역은 어디에 해당하나요?"],
        ["무주택 세대에 대해서 설명해주세요."],
    ],
)
demo.launch()

Running on local URL:  http://127.0.0.1:7862

To create a public link, set `share=True` in `launch()`.


검색된 문서 개수: 8
문서 1 관련성: no
문서 47 관련성: no
문서 7 관련성: no
문서 36 관련성: no
문서 39 관련성: no
문서 18 관련성: no
문서 44 관련성: no
문서 49 관련성: no
관련 문서 개수: 0
검색된 문서 개수: 8
문서 21 관련성: yes
문서 45 관련성: yes
문서 30 관련성: yes
문서 49 관련성: yes
문서 16 관련성: no
문서 1 관련성: no
문서 47 관련성: yes
문서 10 관련성: no
관련 문서 개수: 5
[품질 평가] 불충분 → 폴백 적용 - 답변이 '근거 없음'이라는 불충분한 표현만 포함하고 있어 질문의 핵심을 구체적으로 다루지 않고 있습니다. 문서에서 논리적으로 추론 가능한 답변을 제공하지 않아 충분하지 않습니다.
검색된 문서 개수: 8
문서 1 관련성: yes
문서 2 관련성: no
문서 47 관련성: no
문서 7 관련성: no
문서 6 관련성: no
문서 50 관련성: no
문서 38 관련성: no
문서 48 관련성: no
관련 문서 개수: 1
[품질 평가] 불충분 → 폴백 적용 - 답변이 '근거 없음'으로 구체적인 수원시의 주택건설지역에 대한 정보를 제공하지 않고 있어 질문의 핵심을 다루지 못함.
검색된 문서 개수: 8
문서 1 관련성: yes
문서 2 관련성: no
문서 47 관련성: no
문서 7 관련성: no
문서 6 관련성: no
문서 50 관련성: no
문서 38 관련성: no
문서 48 관련성: no
관련 문서 개수: 1
[품질 평가] 불충분 → 폴백 적용 - 답변이 '근거 없음'으로 구체적인 수원시의 주택건설지역에 대한 정보를 제공하지 않고 있어 질문의 핵심을 다루지 못함.


In [13]:
# Gradio 인터페이스 종료
demo.close()

Closing server running on port: 7862
